# Produce the training data for a detector

A detector belongs to the model it scores, so training one for **your** model starts by
recording how that model answers: you bring the questions, and it records the answers *with
the token log-probabilities behind them* and a verdict on each one.

This notebook produces both, against any OpenAI-compatible endpoint.

1. **Bring the questions** — yours, written into the notebook. A hundred real TriviaQA
   rows are there to run as-is; replace them with the questions your users actually ask.
2. **Answer them**, keeping `top_logprobs`. The distribution behind the answer is what the
   detector reads; the answer text is only used to judge it.
3. **Judge the answers** against their gold answers, with the model as judge.
4. **Fit and evaluate** a WEPR detector on the result, and save it.

**No GPU.** Every model call goes to the endpoint; everything this notebook runs locally is
bookkeeping and a logistic regression.

**Nothing is downloaded.** The questions are a list in the notebook, the answers and the
verdicts come from the endpoint, and the only local work is a logistic regression. There is
no dataset to fetch and no corpus to stage.

## What it writes

Two files, joined on `custom_id`:

| File | Contents |
|---|---|
| `responses.jsonl` | The answers, with `top_logprobs` per token |
| `judgments.jsonl` | The judge's reply for each answer |

Both are the **OpenAI Batch output shape** — one JSON object per line, each wrapping a chat
completion under `custom_id`. That format is not this notebook's invention and not its
private convention: it is what the Batch API returns, so these files are readable by
anything that reads a batch, and can be produced by anything that writes one. Keeping to it
is why nothing downstream needs a conversion step.

Writing them out at all is the point. Generating and judging cost money and time; fitting
costs seconds. With the files on disk you can refit at a different `k`, on `epr` instead of
`wepr`, or against relabelled verdicts, without paying for any of it twice.

## Prerequisites

```bash
uv pip install "artefactual[adapters]"
```

| Variable | Required | What it is |
|---|---|---|
| `OPENAI_BASE_URL` | yes | Any OpenAI-compatible endpoint returning `top_logprobs` |
| `OPENAI_API_KEY` | yes | Its key |
| `OPENAI_MODEL` | yes | The model being scored — the detector you train belongs to it, and the id has to be one your endpoint serves |

**The endpoint has to return at least `K` ranks per token**, which is the one requirement
worth checking before you start. OpenAI's own API accepts `top_logprobs` up to 20, and a
self-hosted vLLM up to its `--max-logprobs` (20 by default), so `K = 15` fits both.
Providers that cap lower, or omit `logprobs` entirely, are refused by name in step 2 rather
than silently training on narrower data.

**A detector belongs to the model it was trained on.** Its weights read that model's
confidence, so scoring a different model with them is not supported; retrain instead.

This notebook is not executed when the documentation is built, because it generates against
a live endpoint. The numbers you see are the ones your run produces.

In [ ]:
# On Colab, uncomment to install the package.
# !pip install -q 'artefactual[adapters]'

## Configuration

Every knob the run has, in one place: change them here and rerun from the top. Nothing
below this cell hard-codes a value that belongs in it.

In [ ]:
import contextlib
import json
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# No default: a model id only means something to the endpoint serving it, and a wrong one
# fails on every request in step 2 rather than here.
MODEL = os.environ["OPENAI_MODEL"]

# Ranks kept per token. Part of the feature definition, not a batch size: WEPR fits one
# coefficient per rank, so a detector is only ever used at the k it was fitted at. Every
# published detector uses 15.
K = 15
# The API caps it at 20, and an endpoint asked for more rejects every request in step 2 --
# which arrives as "no answers were generated" three cells later, pointing at the wrong
# thing.
assert 1 <= K <= 20, "top_logprobs must be between 1 and 20"

SEED = 42
WORKERS = 8

RESPONSES = Path("responses.jsonl")
JUDGMENTS = Path("judgments.jsonl")
# The generation prompt, the paper's (4.1.2). Short answers on purpose: the detector reads
# the token distribution behind the answer, so a model that pads with hedging spends its
# tokens on text that carries nothing to score.
GENERATE = """You are a useful assistant that help finding short and precise answers for a given query or question.
            Please keep your output AS SHORT AND CONCISE AS POSSIBLE.
            Here is the query :
            {query}
            """

# The judging prompt, the paper's, in full. Its fields are `{{name}}` rather than the `{}`
# `str.format` reads, because the reply format it demands is itself a JSON object: `format`
# would take those braces for fields, and so would a question containing one. Step 3 fills
# it by literal replacement.
JUDGE = """You are an expert evaluator tasked with determining if two answers convey compatible information. Your task is to make a binary True/False judgment on whether the answers are SEMANTICALLY COMPATIBLE.

Query:
{{query}}

Expected Answer:
{{expected_answer}}
{{aliases_block}}
Generated Answer:
{{generated_answer}}

CRITICAL INSTRUCTIONS:
1. FIRST, perform a simple VERBATIM TEXT COMPARISON:
   - If the generated answer is IDENTICAL (exact same text) to EITHER the expected answer OR ANY of the answer aliases, your judgment MUST be TRUE
   - If not identical to any of them, proceed to semantic comparison

2. For SEMANTIC COMPARISON, use these MANDATORY RULES:
   - Judge "True" if the generated answer matches the SEMANTIC MEANING of EITHER the expected answer OR ANY of the answer aliases
   - Judge "True" WHENEVER the general meaning or core concept is the same as either the expected answer or any alias
   - Judge "True" if one answer is GENERAL and one is SPECIFIC about the same thing
   - Judge "True" if one answer names a CATEGORY (e.g., "missionaries") and the other provides SPECIFIC INSTANCES of that category (e.g., "Augustine was sent by Pope Gregory")
   - Judge "True" if one answer gives a BRIEF fact and the other ELABORATES with more details
   - Judge "True" if one answer is more detailed but does NOT contradict the other
   - Judge "False" ONLY if the answers directly CONTRADICT all of the expected answer and all aliases, or discuss ENTIRELY different topics

3. EXTREMELY IMPORTANT RULES ABOUT SPECIFICITY:
   - When one answer is general and one is specific → TRUE
   - When one uses a category term and one gives examples → TRUE
   - When one gives "who/what" and the other adds "when/where/how/why" → TRUE
   - When one gives a person's role and the other gives their name → TRUE
   - When one refers to a group and the other names individuals → TRUE

4. Always check if the specific answer is an INSTANCE or EXAMPLE of the general answer
   - If it is, the judgment MUST be TRUE regardless of how detailed the specific answer is

5. The query is provided ONLY for context - do NOT use it in your judgment

6. IMPORTANT: The generated answer should be considered TRUE if it matches EITHER the expected answer OR ANY of the answer aliases in meaning

FINAL CHECK BEFORE SUBMITTING:
- If the generated answer could reasonably be considered matching ANY of the expected answer or aliases → TRUE
- If after reading all answers, they feel like they're talking about the same basic concept → TRUE
- If you think "the generated answer is not contradicting the expected answer or any of its aliases" → TRUE

Your response MUST follow this format:
{
  "judgment": true/false,
  "explanation": "One clear sentence explaining why the answers are compatible or contradictory."
}"""

## Step 1 — bring the questions

**This is the cell to replace.** The questions below are a hundred real TriviaQA rows, kept
so the notebook runs end to end out of the box; the detector you actually want is trained on
the questions your own users ask, because a detector reads how a model behaves on the
traffic it will meet. The paper measured that gap: its numbers drop 10-20 points when a
TriviaQA-trained detector is pointed at WebQuestions.

Two fields per question, and a third that is optional:

| Field | |
|---|---|
| `question` | The text to ask |
| `short_answer` | What a correct reply has to agree with |
| `answer_aliases` | Other wordings that also count, shown to the judge |

Two properties decide whether a question set is usable, and neither is about its fields:
answers must be **short enough for a judge to grade** against `short_answer`, and the model
must get **enough of them wrong** that both classes appear. A model that answers everything
correctly leaves the fit nothing to learn from.

A hundred is the working size: the fit needs both classes and enough of the rarer one to put
a few on the far side of the split, and the check after judging spells out the floor. At
twenty-five a capable model often gets nothing wrong, and the run would spend the requests
and then refuse to fit.

`question_id` is not among the fields because the notebook assigns it, from each question's
position. It is what travels — it becomes `custom_id` on the answers and the verdicts, and
that is what every later join pairs on — so it has to be unique, and deriving it from
position is how that is guaranteed rather than asserted.


In [ ]:
QUESTIONS = [
    {
        "question": "Which Lloyd Webber musical premiered in the US on 10th December 1993?",
        "short_answer": "Sunset Boulevard",
        "answer_aliases": ["Sunset Blvd", "Sunset Blvd.", "Sunset Bulevard", "West Sunset Boulevard"],
    },
    {
        "question": "Who wrote the novel 'Things Fall Apart'?",
        "short_answer": "Chinua Achebe",
        "answer_aliases": ["Achebe"],
    },
    {"question": "What is the capital of Mongolia?", "short_answer": "Ulaanbaatar", "answer_aliases": ["Ulan Bator"]},
    {"question": "Which element has the atomic number 79?", "short_answer": "Gold", "answer_aliases": ["Au"]},
    {"question": "In which year did the Berlin Wall fall?", "short_answer": "1989"},
    {
        "question": "Who painted 'The Garden of Earthly Delights'?",
        "short_answer": "Hieronymus Bosch",
        "answer_aliases": ["Bosch"],
    },
    {
        "question": "What is the longest river in Asia?",
        "short_answer": "Yangtze",
        "answer_aliases": ["Yangtze River", "Chang Jiang"],
    },
    {
        "question": "Who composed 'The Rite of Spring'?",
        "short_answer": "Igor Stravinsky",
        "answer_aliases": ["Stravinsky"],
    },
    {
        "question": "What is the smallest country in the world by area?",
        "short_answer": "Vatican City",
        "answer_aliases": ["the Vatican"],
    },
    {"question": "Which planet is known as the Red Planet?", "short_answer": "Mars"},
    {
        "question": "Who developed the polio vaccine first licensed in 1955?",
        "short_answer": "Jonas Salk",
        "answer_aliases": ["Salk"],
    },
    {"question": "What is the currency of Sweden?", "short_answer": "Krona", "answer_aliases": ["Swedish krona"]},
    {
        "question": "Which sea separates Europe and Africa?",
        "short_answer": "Mediterranean Sea",
        "answer_aliases": ["the Mediterranean"],
    },
    {
        "question": "Who wrote 'One Hundred Years of Solitude'?",
        "short_answer": "Gabriel Garcia Marquez",
        "answer_aliases": ["Garcia Marquez"],
    },
    {"question": "What is the hardest naturally occurring substance?", "short_answer": "Diamond"},
    {
        "question": "Which country hosted the 1992 Summer Olympics?",
        "short_answer": "Spain",
        "answer_aliases": ["Barcelona, Spain"],
    },
    {"question": "What is the chemical symbol for potassium?", "short_answer": "K"},
    {"question": "Who directed the film 'Rashomon'?", "short_answer": "Akira Kurosawa", "answer_aliases": ["Kurosawa"]},
    {
        "question": "What is the largest desert in the world?",
        "short_answer": "Antarctic Desert",
        "answer_aliases": ["Antarctica"],
    },
    {
        "question": "Who was the first woman to win a Nobel Prize?",
        "short_answer": "Marie Curie",
        "answer_aliases": ["Curie"],
    },
    {
        "question": "Which language has the most native speakers?",
        "short_answer": "Mandarin Chinese",
        "answer_aliases": ["Mandarin"],
    },
    {
        "question": "What is the tallest mountain in Africa?",
        "short_answer": "Kilimanjaro",
        "answer_aliases": ["Mount Kilimanjaro"],
    },
    {
        "question": "Who wrote 'The Second Sex'?",
        "short_answer": "Simone de Beauvoir",
        "answer_aliases": ["de Beauvoir"],
    },
    {
        "question": "In which city is the Hermitage Museum?",
        "short_answer": "Saint Petersburg",
        "answer_aliases": ["St Petersburg"],
    },
    {
        "question": "What is the boiling point of water at sea level in Celsius?",
        "short_answer": "100",
        "answer_aliases": ["100 degrees"],
    },
    {
        "question": "Who invented the World Wide Web?",
        "short_answer": "Tim Berners-Lee",
        "answer_aliases": ["Berners-Lee"],
    },
    {"question": "What is the largest island in the Mediterranean?", "short_answer": "Sicily"},
    {
        "question": "Which artist cut off part of his own ear?",
        "short_answer": "Vincent van Gogh",
        "answer_aliases": ["van Gogh"],
    },
    {"question": "What is the study of fungi called?", "short_answer": "Mycology"},
    {"question": "Which country is home to the Great Barrier Reef?", "short_answer": "Australia"},
    {"question": "Who wrote the play 'A Doll's House'?", "short_answer": "Henrik Ibsen", "answer_aliases": ["Ibsen"]},
    {
        "question": "What is the largest organ of the human body?",
        "short_answer": "Skin",
        "answer_aliases": ["the skin"],
    },
    {
        "question": "Which war ended with the Treaty of Versailles?",
        "short_answer": "World War I",
        "answer_aliases": ["the First World War", "WWI"],
    },
    {"question": "What is the capital of New Zealand?", "short_answer": "Wellington"},
    {"question": "Who discovered penicillin?", "short_answer": "Alexander Fleming", "answer_aliases": ["Fleming"]},
    {
        "question": "Which is the deepest ocean trench?",
        "short_answer": "Mariana Trench",
        "answer_aliases": ["the Marianas Trench"],
    },
    {"question": "Who wrote 'Beloved'?", "short_answer": "Toni Morrison", "answer_aliases": ["Morrison"]},
    {"question": "What is the national sport of Japan?", "short_answer": "Sumo", "answer_aliases": ["sumo wrestling"]},
    {"question": "Which gas makes up most of Earth's atmosphere?", "short_answer": "Nitrogen"},
    {
        "question": "Who was the first person to reach the South Pole?",
        "short_answer": "Roald Amundsen",
        "answer_aliases": ["Amundsen"],
    },
    {"question": "What is the largest mammal?", "short_answer": "Blue whale", "answer_aliases": ["the blue whale"]},
    {"question": "Which city is known as the Eternal City?", "short_answer": "Rome"},
    {"question": "Who wrote 'The Wealth of Nations'?", "short_answer": "Adam Smith"},
    {
        "question": "What is the freezing point of water in Fahrenheit?",
        "short_answer": "32",
        "answer_aliases": ["32 degrees"],
    },
    {
        "question": "Which instrument measures atmospheric pressure?",
        "short_answer": "Barometer",
        "answer_aliases": ["a barometer"],
    },
    {"question": "Who painted the ceiling of the Sistine Chapel?", "short_answer": "Michelangelo"},
    {"question": "What is the capital of Canada?", "short_answer": "Ottawa"},
    {"question": "Which metal is liquid at room temperature?", "short_answer": "Mercury"},
    {"question": "Who wrote 'Invisible Man'?", "short_answer": "Ralph Ellison", "answer_aliases": ["Ellison"]},
    {
        "question": "What is the longest bone in the human body?",
        "short_answer": "Femur",
        "answer_aliases": ["the femur", "thigh bone"],
    },
    {
        "question": "Which ocean lies between Africa and Australia?",
        "short_answer": "Indian Ocean",
        "answer_aliases": ["the Indian Ocean"],
    },
    {
        "question": "Who wrote 'Crime and Punishment'?",
        "short_answer": "Fyodor Dostoevsky",
        "answer_aliases": ["Dostoevsky"],
    },
    {"question": "What is the capital of Peru?", "short_answer": "Lima"},
    {
        "question": "Which vitamin is produced when skin is exposed to sunlight?",
        "short_answer": "Vitamin D",
        "answer_aliases": ["D"],
    },
    {
        "question": "Who was the first president of the United States?",
        "short_answer": "George Washington",
        "answer_aliases": ["Washington"],
    },
    {
        "question": "What is the chemical formula for table salt?",
        "short_answer": "NaCl",
        "answer_aliases": ["sodium chloride"],
    },
    {
        "question": "Which composer wrote the 'Moonlight Sonata'?",
        "short_answer": "Beethoven",
        "answer_aliases": ["Ludwig van Beethoven"],
    },
    {"question": "What is the largest planet in the solar system?", "short_answer": "Jupiter"},
    {"question": "Who wrote 'Mrs Dalloway'?", "short_answer": "Virginia Woolf", "answer_aliases": ["Woolf"]},
    {"question": "Which country invented paper?", "short_answer": "China"},
    {"question": "What is the capital of Morocco?", "short_answer": "Rabat"},
    {
        "question": "Who formulated the theory of general relativity?",
        "short_answer": "Albert Einstein",
        "answer_aliases": ["Einstein"],
    },
    {
        "question": "Which bird cannot fly and is native to New Zealand?",
        "short_answer": "Kiwi",
        "answer_aliases": ["the kiwi"],
    },
    {
        "question": "What is the main ingredient in guacamole?",
        "short_answer": "Avocado",
        "answer_aliases": ["avocados"],
    },
    {
        "question": "Who wrote 'The Old Man and the Sea'?",
        "short_answer": "Ernest Hemingway",
        "answer_aliases": ["Hemingway"],
    },
    {"question": "Which planet has the Great Red Spot?", "short_answer": "Jupiter"},
    {"question": "What is the capital of Egypt?", "short_answer": "Cairo"},
    {"question": "Who was the ancient Greek god of the sea?", "short_answer": "Poseidon"},
    {"question": "Which country has the most time zones?", "short_answer": "France"},
    {
        "question": "What does DNA stand for?",
        "short_answer": "Deoxyribonucleic acid",
        "answer_aliases": ["deoxyribonucleic"],
    },
    {"question": "Who wrote 'Pride and Prejudice'?", "short_answer": "Jane Austen", "answer_aliases": ["Austen"]},
    {"question": "What is the smallest prime number?", "short_answer": "2", "answer_aliases": ["two"]},
    {"question": "Which city hosted the first modern Olympic Games?", "short_answer": "Athens"},
    {"question": "What is the capital of Argentina?", "short_answer": "Buenos Aires"},
    {"question": "Who invented the telephone?", "short_answer": "Alexander Graham Bell", "answer_aliases": ["Bell"]},
    {
        "question": "Which is the longest river in South America?",
        "short_answer": "Amazon",
        "answer_aliases": ["the Amazon"],
    },
    {"question": "What is the study of earthquakes called?", "short_answer": "Seismology"},
    {"question": "Who wrote 'Don Quixote'?", "short_answer": "Miguel de Cervantes", "answer_aliases": ["Cervantes"]},
    {"question": "Which metal is the best conductor of electricity?", "short_answer": "Silver"},
    {"question": "What is the capital of Vietnam?", "short_answer": "Hanoi"},
    {"question": "Who directed 'Seven Samurai'?", "short_answer": "Akira Kurosawa", "answer_aliases": ["Kurosawa"]},
    {"question": "Which planet is closest to the Sun?", "short_answer": "Mercury"},
    {
        "question": "What is the largest lake in Africa?",
        "short_answer": "Lake Victoria",
        "answer_aliases": ["Victoria"],
    },
    {"question": "Who wrote 'Frankenstein'?", "short_answer": "Mary Shelley", "answer_aliases": ["Shelley"]},
    {"question": "What is the currency of Japan?", "short_answer": "Yen", "answer_aliases": ["the yen"]},
    {
        "question": "Which mountain range separates Europe and Asia?",
        "short_answer": "Ural Mountains",
        "answer_aliases": ["the Urals"],
    },
    {"question": "Who painted 'Guernica'?", "short_answer": "Pablo Picasso", "answer_aliases": ["Picasso"]},
    {"question": "What is the capital of Norway?", "short_answer": "Oslo"},
    {"question": "Which blood type is the universal donor?", "short_answer": "O negative", "answer_aliases": ["O-"]},
    {"question": "Who wrote 'Waiting for Godot'?", "short_answer": "Samuel Beckett", "answer_aliases": ["Beckett"]},
    {"question": "What is the tallest waterfall in the world?", "short_answer": "Angel Falls"},
    {"question": "Which country is Machu Picchu in?", "short_answer": "Peru"},
    {"question": "What is the chemical symbol for iron?", "short_answer": "Fe"},
    {"question": "Who composed 'The Four Seasons'?", "short_answer": "Antonio Vivaldi", "answer_aliases": ["Vivaldi"]},
    {"question": "What is the capital of Kenya?", "short_answer": "Nairobi"},
    {
        "question": "Which sense is most closely linked to memory?",
        "short_answer": "Smell",
        "answer_aliases": ["olfaction"],
    },
    {"question": "Who wrote 'The Trial'?", "short_answer": "Franz Kafka", "answer_aliases": ["Kafka"]},
    {
        "question": "What is the largest bone in the human foot?",
        "short_answer": "Calcaneus",
        "answer_aliases": ["heel bone"],
    },
    {"question": "Which country produces the most coffee?", "short_answer": "Brazil"},
    {"question": "What is the capital of Portugal?", "short_answer": "Lisbon", "answer_aliases": ["Lisboa"]},
]

# One id per question, from its position: the join key is this notebook's to mint, and a
# duplicate would pair an answer with another question's gold answer -- a wrong label rather
# than an error. Positions cannot collide, so there is nothing left to check.
questions = [
    {
        "question_id": f"q{position:03d}",
        "question": entry["question"],
        "short_answer": entry["short_answer"],
        "answer_aliases": entry.get("answer_aliases") or [],
    }
    for position, entry in enumerate(QUESTIONS)
]

incomplete = [q for q in questions if not q["question"] or not q["short_answer"]]
assert not incomplete, (
    f"{len(incomplete)} question(s) are missing `question` or `short_answer`, e.g. "
    f"{incomplete[0]}. Both are needed: one is asked, the other is what the judge grades against."
)

print(f"{len(questions)} questions")
print(json.dumps(questions[0], indent=2, ensure_ascii=False))

## Step 2 — generate the answers, keeping the log-probabilities

`logprobs=True` and `top_logprobs=K` are what make a response scoreable at all: the
detector reads the token distribution behind the answer, not the answer. A response
generated without them is a valid completion carrying nothing to score, and one generated
with fewer than `K` ranks is refused when it reaches the parser rather than zero-filled —
the missing ranks are unfetched rather than absent, so filling them with zeros would score
the answer as more confident than it was. Generating *wider* than `K` is safe; surplus
ranks are dropped.

Prompt and sampling follow the paper (§4.1.2): non-greedy at `T = 1.0`, `top_p = 1.0`.
The paper also sets `top_k = 50`, which is not an OpenAI parameter — so this samples at
whatever the endpoint's default is, and the distribution is not quite the paper's. It also
sends `max_completion_tokens`, which some OpenAI-compatible servers still only accept as
`max_tokens`; if every request fails, that is the first thing to check.
Non-greedy is the point — the method measures hesitation in the raw distribution.

Requests are threaded because the round trips, not the fitting, are what make this slow. A
failure is returned rather than raised and is written out as an error row, exactly as a
batch job records one, so the count is visible rather than silently missing.

In [ ]:
from openai import OpenAI, OpenAIError

# `max_retries` above the SDK's default of 2: this fires N requests at once, and a burst
# of 429s that exhausts the retries becomes a thinner dataset rather than an error.
client = OpenAI(max_retries=6)  # reads OPENAI_BASE_URL and OPENAI_API_KEY

# Which endpoint this is actually talking to. Unset, OPENAI_BASE_URL silently means
# api.openai.com, and a self-hosted run then fails N times with an authentication error.
print(f"endpoint: {client.base_url}")


def generate(question):
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": GENERATE.format(query=question["question"])}],
            logprobs=True,
            top_logprobs=K,
            temperature=1.0,
            top_p=1.0,
            max_completion_tokens=200,
        )
    except OpenAIError as error:
        # Every failure this call can produce -- transport, timeout, rate limit, a
        # rejected request -- is an OpenAIError, and one of them should not cost the
        # run. Anything else is a bug in the code above and should not be caught here.
        return error


with RESPONSES.open("w", encoding="utf-8") as out, ThreadPoolExecutor(max_workers=WORKERS) as pool:
    # Written as each result arrives, not after the pool finishes. `generate` returns the
    # API's failures rather than raising them, but the SDK does not wrap every transport
    # pathology -- a 502 whose HTML body arrives under a JSON content type raises
    # JSONDecodeError from inside the worker -- and that would come out of `pool.map` and
    # discard every answer already paid for. This way the file holds what was generated up
    # to the failure, which is the whole argument for writing it out at all.
    generated = []
    for question, result in zip(questions, pool.map(generate, questions), strict=True):
        generated.append(result)
        failed = isinstance(result, Exception)
        out.write(
            json.dumps(
                {
                    "id": f"chatcmpl-{question['question_id']}",
                    "custom_id": question["question_id"],
                    "response": None if failed else {"status_code": 200, "body": result.model_dump()},
                    # The class name, not the provider's text: an authentication error
                    # quotes the key it rejected, and this file is one you hand onward.
                    # The full message is printed below, where it stays in the session.
                    "error": {"message": type(result).__name__} if failed else None,
                },
                # ASCII-escaped: every reader of these files splits them with
                # `splitlines()`, which breaks on U+2028, U+2029 and U+0085 -- characters
                # JSON does not require escaping and a model can emit.
                ensure_ascii=True,
            )
            + "\n"
        )
ok = [(q, r) for q, r in zip(questions, generated) if not isinstance(r, Exception)]
print(f"wrote {RESPONSES}, {len(ok)}/{len(generated)} generated")
for question, result in zip(questions, generated):
    if isinstance(result, Exception):
        print(f"  failed: {question['question_id']}: {result}")

In [ ]:
# Nothing came back at all -- almost always OPENAI_BASE_URL, the key, or a model name the
# endpoint does not serve. Checked before indexing, because the errors above say what
# happened and a bare IndexError here would not.
assert ok, (
    "no answers were generated. Check OPENAI_BASE_URL, OPENAI_API_KEY and OPENAI_MODEL, "
    "and read the per-request errors above: an endpoint that rejects `top_logprobs`, "
    "`temperature` or `max_completion_tokens` fails every request the same way."
)

# The cheapest place to notice an endpoint that ignored `top_logprobs`: it returns a
# perfectly valid completion carrying nothing to score. `logprobs` is checked before
# `.content` because that is the shape the failure actually takes -- reaching straight for
# `.content` raises an AttributeError that names nothing useful.
# `.content` is optional inside `logprobs` as well, and a provider that returns the
# object with nothing in it fails the same way for the reader.
missing = [q["question_id"] for q, r in ok if not (r.choices[0].logprobs and r.choices[0].logprobs.content)]
assert not missing, (
    f"{len(missing)} response(s) carry no logprobs at all, e.g. {missing[:3]}. "
    f"The endpoint accepted `logprobs=True` and ignored it; it cannot be used at any k."
)

# Every token, not just the first -- a response whose later tokens are narrower would pass
# a first-token check and fail inside `fit`.
widths = [len(t.top_logprobs) for _, r in ok for t in (r.choices[0].logprobs.content or [])]
assert widths and min(widths) >= K, (
    f"endpoint returned {min(widths) if widths else 0} ranks per token, need {K}; "
    f"raise top_logprobs, or lower K and fit at that rank count"
)

print(f"{min(widths)}-{max(widths)} ranks per token across {len(ok)} answers")
print(f"{ok[0][0]['question']}\n  -> {(ok[0][1].choices[0].message.content or '').strip()}")

## Step 3 — judge the answers

The judge prompt is the paper's, in full, in the configuration cell at the top. It shows the judge the question,
the gold answer, its aliases and the generated answer, and asks for
`{"judgment": true|false, "explanation": "..."}` -- a semantic comparison rather than a
string match, so an answer that is right but worded differently is accepted.

**The judge answers the opposite question to the one we record.** Its `judgment: true`
means the answer was **correct**; the file we write states `hallucination`, the positive
class the detector predicts. The conversion is `hallucination = not judgment`, done once,
at the point of writing.

Rendering is by literal replacement rather than `str.format`: the prompt ends with a JSON
example whose braces `format` would read as fields, and a question containing a brace would
corrupt everything after it.

Grading is deterministic — `temperature=0` — and a verdict that will not parse is dropped
with its answer rather than guessed at.

In [ ]:
def render_judge(question, completion):
    aliases = question["answer_aliases"]
    # Each alias wrapped in newlines and the block closed with a blank line: that is what
    # the original jinja template emits, with trim_blocks off.
    block = (
        ("\nAnswer Aliases (Additional Correct Answers):\n" + "".join(f"\n- {alias}\n" for alias in aliases) + "\n\n")
        if aliases
        else "\n"
    )
    return (
        JUDGE
        .replace("{{query}}", question["question"])
        .replace("{{expected_answer}}", question["short_answer"])
        .replace("{{aliases_block}}", block)
        .replace("{{generated_answer}}", completion.choices[0].message.content or "")
    )


print(render_judge(ok[0][0], ok[0][1])[:600])

In [ ]:
def judge(pair):
    """The judge's reply, kept whole: the verdict is derived from it, not instead of it."""
    question, completion = pair
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": render_judge(question, completion)}],
            temperature=0,
            max_completion_tokens=200,
        )
    except OpenAIError as error:
        # Every failure this call can produce -- transport, timeout, rate limit, a
        # rejected request -- is an OpenAIError, and one of them should not cost the
        # run. Anything else is a bug in the code above and should not be caught here.
        return error


def read_judgment(content):
    """The judge's verdict as a bool, or None when the reply cannot be read as one.

    A bare `json.loads` is not enough. Models wrap the object in prose or in a ```json
    fence often enough to matter, so an unparsed reply falls back to scanning for the
    token. And only a real boolean counts: `bool("false")` is True, so a judge that emits
    the value as a string -- which a loose schema invites -- would otherwise mark every
    wrong answer correct, silently.
    """
    with contextlib.suppress(json.JSONDecodeError, KeyError, TypeError):
        verdict = json.loads(content)["judgment"]
        if isinstance(verdict, bool):
            return verdict

    lowered = (content if isinstance(content, str) else "").lower()
    if '"judgment": true' in lowered:
        return True
    if '"judgment": false' in lowered:
        return False
    return None


def hallucinated(verdict):
    """`judgment: true` means the answer was CORRECT, so the flag is its negation.

    The one place the judge's convention and the detector's meet. None when the reply
    cannot be read as a verdict at all.
    """
    if isinstance(verdict, Exception):
        return None
    judgment = read_judgment(verdict.choices[0].message.content)
    return None if judgment is None else not judgment


with JUDGMENTS.open("w", encoding="utf-8") as out, ThreadPoolExecutor(max_workers=WORKERS) as pool:
    # Written as each reply arrives, for the same reason step 2 is: a failure the SDK does
    # not wrap comes out of `pool.map`, and the verdicts already paid for should survive it.
    # The verdict stays as the judge wrote it; nothing is distilled out.
    verdicts = []
    for (question, _), verdict in zip(ok, pool.map(judge, ok), strict=True):
        verdicts.append(verdict)
        failed = isinstance(verdict, Exception)
        out.write(
            json.dumps(
                {
                    "id": f"chatcmpl-judge-{question['question_id']}",
                    "custom_id": question["question_id"],
                    "response": None if failed else {"status_code": 200, "body": verdict.model_dump()},
                    "error": {"message": type(verdict).__name__} if failed else None,
                },
                ensure_ascii=True,
            )
            + "\n"
        )
# `judgment: true` means the answer was CORRECT, so the label is its negation. The one
# place the judge's convention and the detector's meet.
labels = [(pair, hallucinated(v)) for pair, v in zip(ok, verdicts)]
judgments = [(question, completion, flag) for (question, completion), flag in labels if flag is not None]

dropped = len(verdicts) - len(judgments)
print(f"wrote {JUDGMENTS}, {len(verdicts)} replies" + (f", {dropped} unreadable and dropped" if dropped else ""))

# A reply cut off at `max_completion_tokens` is invalid JSON, so it lands in `dropped` with
# nothing saying why. The judge answers in JSON *and* explains itself, so this is the cap
# that runs out first.
truncated = sum(1 for v in verdicts if not isinstance(v, Exception) and v.choices[0].finish_reason == "length")
if truncated:
    print(f"  {truncated} reply(ies) hit max_completion_tokens; raise it and rerun step 3")
for question, completion, flag in judgments[:2]:
    said = (completion.choices[0].message.content or "").strip()
    print(f"  [{'hallucination' if flag else 'grounded'}] said {said[:40]!r} (gold: {question['short_answer']!r})")

In [ ]:
import numpy as np

y = np.array([flag for _, _, flag in judgments], dtype=int)

# Both classes are needed, and this is where a run fails cheaply rather than inside `fit`.
# All-correct means the questions were too easy for this model; all-wrong usually means it
# is not answering in the short form the judge expects.
# Enough of the rarer class that the held-out quarter carries a few of it: at TEST_SIZE
# a floor of 12 puts 3 on the far side, and below that a ROC-AUC is arithmetic over a
# handful of rows rather than a measurement. Checked here, before the fit and before
# anything downstream is built on the labels.
rarer = min(y.sum(), len(y) - y.sum())
assert rarer >= 12, (
    f"only {rarer} of {len(y)} answers are in the rarer class, which leaves too few on the "
    f"far side of the split to score. Add questions, or make them harder if "
    f"nothing was hallucinated and easier if everything was."
)

print(f"{len(y)} judged answers, {y.sum()} hallucinations ({y.mean():.0%})")

## Step 4 — fit and evaluate

The two files are written, so from here everything is repeatable for free. This step reads
them back rather than using what is still in memory -- which is what makes it true that you
can come back tomorrow, change `k`, and refit without paying for generation again.

`trainable=True` returns an unfitted pipeline -- parser, entropy reduction, logistic
regression -- that takes the raw responses, so there is no feature extraction to write.

**ROC-AUC** scores the ranking, which governs triage by score and is what the paper
reports; the **classification report** scores the decisions at 0.5, where recall on the
`hallucination` row is the fraction actually flagged. Only the AUC carries over to another
threshold.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from artefactual.scoring import wepr


# Read back from disk, not from the variables above: this is the path a fresh kernel takes,
# and the one anything else reading these files takes too.
def completions(path):
    """Every usable completion in a Batch output file, by `custom_id`.

    `.get` and the blank-line skip because this reads a file, not the variables above: it
    is the same code that would read a file written by anything else.
    """
    found = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        envelope = row.get("response") or {}
        # The completion is always the envelope's `body` -- the only shape the library
        # reads, and the shape written above. A line without one is not a Batch line.
        completion = envelope.get("body") if isinstance(envelope, dict) else None
        if row.get("error") is not None or not isinstance(completion, dict):
            continue
        found[row["custom_id"]] = completion
    return found


answers = completions(RESPONSES)
labelled, unreadable = [], 0
for custom_id, completion in completions(JUDGMENTS).items():
    if custom_id not in answers:
        continue
    # The same reader step 3 used. A fenced or prose-wrapped reply is the common case, and
    # a bare `json.loads` here would end the run on the last cell that does anything.
    judgment = read_judgment(completion["choices"][0]["message"]["content"])
    if judgment is None:
        unreadable += 1
        continue
    # `judgment: true` means the answer was correct, so the label is its negation.
    labelled.append((answers[custom_id], int(not judgment)))

if unreadable:
    print(f"{unreadable} verdict(s) could not be read and are not trained on")

responses = [completion for completion, _ in labelled]
y = np.array([label for _, label in labelled])
print(f"read {len(responses)} labelled answers back from {RESPONSES.name} and {JUDGMENTS.name}")

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)
detector = wepr(k=K, trainable=True).fit(x_train, y_train)
print(f"fitted on {len(y_train)}, holding out {len(y_test)}")

scores = detector.predict_proba(x_test)[:, 1]
print(f"\nROC-AUC: {roc_auc_score(y_test, scores):.2f}")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

In [ ]:
path = detector.save_estimator("wepr-generated.skops")
print(f"wrote {path}")

reloaded = wepr(path, k=K)

# Held-out answers: the rows the fit above never saw. `responses` holds the completions as
# they were read back from the file, so they are plain dicts rather than SDK objects.
for completion, label in list(zip(x_test, y_test))[:5]:
    probability = reloaded.predict_proba(completion)[0, 1]
    said = completion["choices"][0]["message"]["content"].strip()
    print(f"[{'hallucination' if label else 'grounded    '}] P={probability:.3f}  said {said[:40]!r}")

## Where to go next

- **Refit without regenerating.** The two files are the expensive part: a different `k`,
  `epr` instead of `wepr`, or relabelled verdicts all reuse them, and none of it costs
  another request.
- **Feed the CLI instead.** This run's two Batch files go straight into the
  `train_detector.py` script in
  [the repository](https://github.com/artefactory/artefactual/blob/main/scripts/train_detector.py),
  which also reports the bootstrap confidence intervals the fit above does not — worth
  having, because a holdout this size cannot pin a score down on its own.
- **More questions.** The generation is the cost and the fit is seconds, so lengthen
  `QUESTIONS` rather than economising on labels.
- **At thousands of questions**, stop making one request per answer. Batch submission
  takes a JSONL of requests and returns the JSONL these files already are, at roughly half
  the price: OpenAI's [Batch API](https://platform.openai.com/docs/api-reference/batch) if
  your provider hosts one, or an offline batch runner against a self-hosted server. Only
  steps 2 and 3 change -- what they write, and everything after it, stays as it is.
- **Your own questions.** Only step 1 changes, and it is a list. Training data should
  resemble the traffic being scored: the paper's numbers drop 10-20 points when a
  TriviaQA-trained detector meets WebQuestions.
